In [1]:
import pandas as pd, numpy as np, os, sys, torch, torch.nn as nn
sys.path.insert(0, "..")  # ensure current directory is in the path
from base_splits import build_base_splits, load_splits, save_splits, summarize_splits, load_dataframe
from sklearn.preprocessing import StandardScaler
from models.lstm_xgb import LSTM_XGB

scaler = StandardScaler()

splits = load_splits(path="../base_splits.pkl")  
print(summarize_splits(splits))

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

cols = splits[0].train.columns[1:-1]

Loaded base splits for 8 classes from /home/maddie/cd-study/Stages/base_splits.pkl
   label  train_n  val_n  test_n  time_start   time_end
0      0     1000    300     300    3.993639   4.153525
1      1     1000    300     300    9.156422   9.316307
2      2     1000    300     300    4.507887   4.667771
3      3     1000    300     300    4.866492   5.026376
4      4     1000    300     300    1.528859   1.688743
5      5     1000    300     300    8.582310   8.742194
6      6     1000    300     300    8.765543   8.925427
7      7     1000    300     300   10.680525  10.840409
device: NVIDIA A30


In [2]:
splits[0].train

,Time,Ipv,Vpv,Vdc,ia,ib,ic,va,vb,vc,Iabc,If,Vabc,Vf,Fault
0,3.993639,2.381195,90.435791,146.755119,-0.517823,-0.174561,0.598384,108.996277,37.019196,-149.659932,0.667383,49.864805,154.852131,49.993418,0
1,3.993739,2.203339,90.441895,146.755163,-0.464112,-0.201416,0.605098,107.164001,42.094116,-150.801086,0.667383,49.864805,154.852131,49.993418,0
2,3.993839,2.184418,90.527344,147.048177,-0.484254,-0.214844,0.605098,103.294525,46.638641,-151.640879,0.667418,49.855768,154.848470,49.993129,0
3,3.993939,2.279022,90.588379,146.755252,-0.417115,-0.241699,0.618525,98.810272,51.460419,-151.576589,0.667418,49.855768,154.848470,49.993129,0
4,3.994039,2.384979,90.527344,146.755297,-0.437257,-0.248413,0.605098,94.506836,55.872345,-152.822215,0.667418,49.855768,154.848470,49.993129,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,4.093131,2.451202,90.252686,146.506421,-0.531251,-0.080566,0.551387,125.643463,14.489441,-140.538737,0.666207,50.116544,154.762097,49.996167,0
996,4.093231,2.456879,90.319824,146.506465,-0.531251,-0.114136,0.551387,122.352600,17.707977,-141.551310,0.666044,50.120195,154.757574,49.996665,0
997,4.093331,2.222260,90.319824,146.506510,-0.504396,-0.127563,0.564814,121.098938,22.831116,-143.897909,0.666044,50.120195,154.757574,49.996665,0
998,4.093431,2.201447,90.454102,146.799523,-0.511109,-0.167847,0.564814,116.530304,27.363586,-146.429342,0.666044,50.120195,154.757574,49.996665,0


In [ ]:
# Z-scale
dct = dict()
scaler.fit(splits[0].train[cols])
for i in range(len(splits)):
    dct[i] = dict()
    dct[i].update({
        "train": pd.DataFrame(scaler.transform(splits[i].train[cols]), columns=cols, index=splits[i].train.index),
        "val": pd.DataFrame(scaler.transform(splits[i].val[cols]), columns=cols, index=splits[i].val.index),
        "test": pd.DataFrame(scaler.transform(splits[i].test[cols]), columns=cols, index=splits[i].test.index),
        
    })


In [4]:
dct

{0: {'train':           Ipv       Vpv       Vdc        ia        ib        ic        va  \
  0    0.764559 -0.246398 -0.104465 -1.087334 -0.362383  1.421336  0.997229   
  1   -1.030293 -0.225499 -0.104394 -0.973647 -0.417662  1.436039  0.980493   
  2   -1.221234  0.067085  0.362327 -1.016280 -0.445302  1.436039  0.945150   
  3   -0.266526  0.276074 -0.104252 -0.874171 -0.500581  1.465446  0.904192   
  4    0.802748  0.067085 -0.104181 -0.916803 -0.514401  1.436039  0.864885   
  ..        ...       ...       ...       ...       ...       ...       ...   
  995  1.471044 -0.873364 -0.500598 -1.115756 -0.168906  1.318414  1.149280   
  996  1.528326 -0.643476 -0.500527 -1.115756 -0.238005  1.318414  1.119222   
  997 -0.839351 -0.643476 -0.500456 -1.058913 -0.265644  1.347821  1.107772   
  998 -1.049387 -0.183701 -0.033736 -1.073124 -0.348563  1.347821  1.066043   
  999  0.287205  0.547759 -0.966964 -0.987858 -0.348563  1.377227  1.038627   
  
             vb        vc      Iabc  

In [ ]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv("../CSV_Files/val.csv").to_numpy()
    test  = pd.read_csv("../CSV_Files/test.csv").to_numpy()
    return to_tensors(train), to_tensors(val), to_tensors(test)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }

In [ ]:
dct.keys()

dict_keys([0, 1, 2, 3, 4, 5, 6, 7])

In [20]:
torch.from_numpy(dct[0]["train"].Ipv.to_numpy().astype("float32"))

tensor([ 7.6456e-01, -1.0303e+00, -1.2212e+00, -2.6653e-01,  8.0275e-01,
         6.1181e-01, -8.1071e-01, -1.0685e+00,  1.6309e-01,  1.1560e+00,
         1.0032e+00, -8.1071e-01, -8.2026e-01, -1.4241e-01,  6.8818e-01,
         3.6358e-01, -8.8709e-01, -1.2308e+00,  1.3374e+00,  1.5379e+00,
         1.0032e+00, -7.3433e-01, -7.6297e-01, -4.8611e-01,  8.6716e-02,
        -1.8302e-02, -1.2403e+00, -1.1162e+00,  3.8981e-02,  1.2419e+00,
         1.0414e+00, -9.6346e-01, -1.1067e+00,  1.0339e-02,  9.0777e-01,
         5.3543e-01, -1.1162e+00, -1.0876e+00,  1.1536e-01,  9.9369e-01,
         6.0226e-01, -1.0207e+00, -1.3740e+00, -1.2332e-01,  8.7912e-01,
         7.9320e-01, -1.0017e+00, -1.4313e+00, -3.9064e-01,  1.0223e+00,
         7.0728e-01, -1.9277e+00, -1.4790e+00, -1.2332e-01,  4.9724e-01,
         1.1655e+00, -1.1162e+00, -1.3072e+00, -2.8562e-01,  1.2419e+00,
         7.3592e-01, -1.1544e+00, -1.3072e+00, -3.9064e-01,  4.9724e-01,
         1.8219e-01, -1.1353e+00, -1.2594e+00,  1.3